# 🧪 A/B Testing — Landing Page Optimization

**Projet 2 — Expérimentation contrôlée**

## Objectif
Déterminer si une nouvelle version d'une landing page (Version B) convertit mieux que la version originale (Version A) en utilisant des méthodes statistiques rigoureuses.

## Plan d'analyse
1. **Chargement & Exploration** — Audit du dataset
2. **Vérification du design** — Équilibre des groupes, SRM check
3. **Tests statistiques** — Chi², Z-test, t-test
4. **Visualisations** — Distributions, bootstrap CI
5. **Analyse segmentée** — Effets hétérogènes
6. **Rapport de décision** — Recommandation finale


In [ ]:
# ─── IMPORTS ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, norm, binom_test
from scipy.stats import bootstrap
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('dark_background')
COLORS = {
    'control': '#6c63ff',
    'treatment': '#00d4aa',
    'bg': '#1a1d26',
    'grid': '#2a2d3e',
    'text': '#8b90a8',
    'danger': '#e74c3c',
    'warning': '#f39c12'
}

print('✅ Imports OK')

## 1. Chargement & Exploration des données


In [ ]:
# ─── CHARGEMENT DES DONNÉES ─────────────────────────────
try:
    df = pd.read_csv('ab_data.csv')
    print(f'✅ Dataset chargé depuis ab_data.csv — {len(df):,} lignes')
except FileNotFoundError:
    print('⚠️  Fichier ab_data.csv non trouvé. Génération du dataset synthétique...')
    
    np.random.seed(42)
    n = 10000
    group = np.random.choice(['control', 'treatment'], size=n, p=[0.5, 0.5])
    conv_rate = np.where(group == 'treatment', 0.127, 0.112)
    converted = np.random.binomial(1, conv_rate)
    time_on_page = np.where(
        group == 'treatment',
        np.random.gamma(shape=3.5, scale=25, size=n),
        np.random.gamma(shape=3.0, scale=22, size=n)
    )
    clicks = np.where(
        group == 'treatment',
        np.random.poisson(lam=4.2, size=n),
        np.random.poisson(lam=3.8, size=n)
    )
    revenue = np.where(
        converted == 1,
        np.random.lognormal(mean=3.5, sigma=0.8, size=n),
        0
    )
    device = np.random.choice(['desktop', 'mobile', 'tablet'], size=n, p=[0.55, 0.35, 0.10])
    country = np.random.choice(['FR', 'US', 'UK', 'DE', 'ES'], size=n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    age_group = np.random.choice(['18-24', '25-34', '35-44', '45-54', '55+'], size=n,
                                   p=[0.15, 0.30, 0.25, 0.18, 0.12])
    timestamps = pd.date_range('2024-01-01', periods=n, freq='1min')
    
    df = pd.DataFrame({
        'user_id': range(1, n+1),
        'timestamp': timestamps,
        'group': group,
        'converted': converted,
        'time_on_page': time_on_page.round(1),
        'clicks': clicks,
        'revenue': revenue.round(2),
        'device': device,
        'country': country,
        'age_group': age_group
    })
    df.to_csv('ab_data.csv', index=False)
    print(f'✅ Dataset synthétique généré et sauvegardé — {len(df):,} lignes')

In [ ]:
# ─── FEATURE ENGINEERING ────────────────────────────────
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['week'] = df['timestamp'].dt.isocalendar().week

print('=== APERÇU DES DONNÉES ===')
print(df.head(10))
print(f'\nShape : {df.shape}')
print(f'\nValeurs manquantes :\n{df.isnull().sum()}')
print(f'\nTypes :\n{df.dtypes}')

In [ ]:
# ─── STATISTIQUES DESCRIPTIVES ──────────────────────────
print('=== STATISTIQUES DESCRIPTIVES ===')
print(df.describe())

print('\n=== DISTRIBUTION DES GROUPES ===')
print(df['group'].value_counts())

print('\n=== TAUX DE CONVERSION PAR GROUPE ===')
print(df.groupby('group')['converted'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'total', 'sum': 'convertis', 'mean': 'taux'}
))

## 2. Vérification du Design Expérimental


In [ ]:
# ─── SRM CHECK (Sample Ratio Mismatch) ──────────────────
# Vérifier que les groupes sont bien équilibrés

n_control = len(df[df['group'] == 'control'])
n_treatment = len(df[df['group'] == 'treatment'])
n_total = len(df)
expected_per_group = n_total / 2

# Test chi² sur la répartition
chi2_srm, p_srm = stats.chisquare([n_control, n_treatment], f_exp=[expected_per_group, expected_per_group])

print('=== SRM CHECK (Sample Ratio Mismatch) ===')
print(f'Contrôle (A) : {n_control:,} users ({n_control/n_total*100:.1f}%)')
print(f'Traitement (B) : {n_treatment:,} users ({n_treatment/n_total*100:.1f}%)')
print(f'Chi² SRM : {chi2_srm:.4f}')
print(f'p-value SRM : {p_srm:.4f}')

if p_srm < 0.05:
    print('⚠️  ALERTE : SRM détecté ! Le ratio n\'est pas équilibré. Investiguer le processus de randomisation.')
else:
    print('✅ Pas de SRM détecté — les groupes sont bien équilibrés.')

## 3. Tests Statistiques


In [ ]:
# ─── PARAMÈTRES DU TEST ─────────────────────────────────
ALPHA = 0.05
POWER_TARGET = 0.80

control = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

conv_control = control['converted'].mean()
conv_treatment = treatment['converted'].mean()

print(f'Seuil alpha : {ALPHA}')
print(f'Taux de conversion Contrôle (A)   : {conv_control*100:.3f}%')
print(f'Taux de conversion Traitement (B) : {conv_treatment*100:.3f}%')
print(f'Lift absolu  : {(conv_treatment - conv_control)*100:+.3f} pp')
print(f'Lift relatif : {(conv_treatment - conv_control)/conv_control*100:+.2f}%')

In [ ]:
# ─── TEST CHI² ──────────────────────────────────────────
contingency = pd.crosstab(df['group'], df['converted'])
print('Table de contingence :')
print(contingency)

chi2_val, p_chi2, dof, expected = chi2_contingency(contingency)
print(f'\n📊 Test du Chi²')
print(f'Statistique χ² = {chi2_val:.4f}')
print(f'p-value        = {p_chi2:.6f}')
print(f'Degrés liberté = {dof}')
print(f'\nConclusion : {"✅ Rejet H₀ — différence SIGNIFICATIVE" if p_chi2 < ALPHA else "❌ Non rejet H₀ — différence NON significative"}')

In [ ]:
# ─── Z-TEST SUR PROPORTIONS ─────────────────────────────
p_pool = df['converted'].mean()
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))
z_score = (conv_treatment - conv_control) / se
p_ztest = 2 * (1 - norm.cdf(abs(z_score)))

# Intervalle de confiance
z_critical = norm.ppf(1 - ALPHA/2)
margin = z_critical * se
diff = conv_treatment - conv_control
ci_low, ci_high = diff - margin, diff + margin

print(f'📊 Z-test sur proportions')
print(f'Z-score    = {z_score:.4f}')
print(f'p-value    = {p_ztest:.6f}')
print(f'Z critique = ±{z_critical:.4f}')
print(f'IC {int((1-ALPHA)*100)}%  = [{ci_low:.5f}, {ci_high:.5f}]')
print(f'\nLe 0 est {"EN DEHORS" if not (ci_low <= 0 <= ci_high) else "DANS"} l\'IC → Différence {"significative" if not (ci_low <= 0 <= ci_high) else "non significative"}')

In [ ]:
# ─── TESTS MÉTRIQUES SECONDAIRES ────────────────────────
print('📊 T-test — Temps sur la page')
t_stat, p_ttest = ttest_ind(control['time_on_page'], treatment['time_on_page'])
print(f't-statistic = {t_stat:.4f} | p-value = {p_ttest:.6f}')
print(f'Contrôle    : μ = {control["time_on_page"].mean():.1f}s, σ = {control["time_on_page"].std():.1f}s')
print(f'Traitement  : μ = {treatment["time_on_page"].mean():.1f}s, σ = {treatment["time_on_page"].std():.1f}s')

print(f'\n📊 Mann-Whitney U — Temps sur la page (non-paramétrique)')
u_stat, p_mwu = mannwhitneyu(control['time_on_page'], treatment['time_on_page'], alternative='two-sided')
print(f'U-statistic = {u_stat:.0f} | p-value = {p_mwu:.6f}')

print(f'\n📊 T-test — Nombre de clics')
t_stat2, p_ttest2 = ttest_ind(control['clicks'], treatment['clicks'])
print(f't-statistic = {t_stat2:.4f} | p-value = {p_ttest2:.6f}')

## 4. Visualisations


In [ ]:
# ─── VIZ 1 : TAUX DE CONVERSION ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0d0f14')
fig.suptitle('A/B Test — Vue d\'ensemble', color='white', fontsize=14, fontweight='bold', y=1.02)

# Barplot conversions
ax = axes[0]
ax.set_facecolor(COLORS['bg'])
groups = ['Contrôle (A)', 'Traitement (B)']
rates = [conv_control * 100, conv_treatment * 100]
bars = ax.bar(groups, rates, color=[COLORS['control'], COLORS['treatment']], width=0.5, zorder=3)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{rate:.2f}%', ha='center', va='bottom', color='white', fontweight='bold')
ax.set_title('Taux de conversion', color='white')
ax.set_ylabel('%', color=COLORS['text'])
ax.set_ylim(0, max(rates) * 1.3)
ax.tick_params(colors=COLORS['text'])
for spine in ax.spines.values(): spine.set_color(COLORS['grid'])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)

# Dist temps
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])
ax2.hist(control['time_on_page'], bins=40, alpha=0.6, color=COLORS['control'], density=True, label='Contrôle')
ax2.hist(treatment['time_on_page'], bins=40, alpha=0.6, color=COLORS['treatment'], density=True, label='Traitement')
ax2.set_title('Temps sur la page (s)', color='white')
ax2.legend(facecolor=COLORS['grid'], labelcolor='white')
ax2.tick_params(colors=COLORS['text'])
for spine in ax2.spines.values(): spine.set_color(COLORS['grid'])
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# Bootstrap distribution
ax3 = axes[2]
ax3.set_facecolor(COLORS['bg'])
np.random.seed(42)
boot_diffs = [np.random.choice(treatment['converted'], 500, replace=True).mean() - 
              np.random.choice(control['converted'], 500, replace=True).mean() 
              for _ in range(1000)]
ax3.hist(boot_diffs, bins=40, color=COLORS['control'], alpha=0.7, density=True)
ax3.axvline(0, color=COLORS['danger'], lw=2, linestyle='--', label='H₀: diff=0')
ax3.axvline(np.mean(boot_diffs), color=COLORS['treatment'], lw=2, linestyle='--', label='Diff observée')
ci_lo_b, ci_hi_b = np.percentile(boot_diffs, [2.5, 97.5])
ax3.axvspan(ci_lo_b, ci_hi_b, alpha=0.1, color=COLORS['treatment'])
ax3.set_title('Bootstrap — Distribution diff.', color='white')
ax3.legend(facecolor=COLORS['grid'], labelcolor='white', fontsize=8)
ax3.tick_params(colors=COLORS['text'])
for spine in ax3.spines.values(): spine.set_color(COLORS['grid'])
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('plots/01_ab_overview.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✅ Graphique sauvegardé : plots/01_ab_overview.png')

In [ ]:
# ─── VIZ 2 : ANALYSE SEGMENTÉE ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0d0f14')
fig.suptitle('Analyse segmentée — Lift par sous-groupe', color='white', fontsize=14, fontweight='bold')

for idx, (col, ax) in enumerate(zip(['device', 'country', 'age_group'], axes)):
    ax.set_facecolor(COLORS['bg'])
    seg_lifts = []
    for val in sorted(df[col].unique()):
        sub = df[df[col] == val]
        c = sub[sub['group'] == 'control']['converted'].mean()
        t = sub[sub['group'] == 'treatment']['converted'].mean()
        lift = (t - c) / c * 100 if c > 0 else 0
        seg_lifts.append((val, lift))
    
    labels, lifts = zip(*seg_lifts)
    colors = [COLORS['treatment'] if l > 0 else COLORS['danger'] for l in lifts]
    ax.barh(labels, lifts, color=colors, zorder=3)
    ax.axvline(0, color='white', lw=1, linestyle='--')
    ax.set_title(col.replace('_', ' ').title(), color='white')
    ax.set_xlabel('Lift (%)', color=COLORS['text'])
    ax.tick_params(colors=COLORS['text'])
    for spine in ax.spines.values(): spine.set_color(COLORS['grid'])
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.xaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('plots/02_segment_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✅ Graphique sauvegardé : plots/02_segment_analysis.png')

In [ ]:
# ─── VIZ 3 : ÉVOLUTION TEMPORELLE ───────────────────────
daily = df.groupby(['date', 'group'])['converted'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0f14')
fig.suptitle('Évolution temporelle', color='white', fontsize=14, fontweight='bold')

ax = axes[0]
ax.set_facecolor(COLORS['bg'])
for grp, color, label in [('control', COLORS['control'], 'Contrôle (A)'),
                            ('treatment', COLORS['treatment'], 'Traitement (B)')]:
    data = daily[daily['group'] == grp].sort_values('date')
    ax.plot(range(len(data)), data['converted'] * 100, color=color, lw=2, label=label)
    ax.fill_between(range(len(data)), data['converted'] * 100, alpha=0.1, color=color)
ax.set_xlabel('Jours', color=COLORS['text'])
ax.set_ylabel('Taux conversion (%)', color=COLORS['text'])
ax.set_title('Convergence des taux', color='white')
ax.legend(facecolor=COLORS['grid'], labelcolor='white')
ax.tick_params(colors=COLORS['text'])
for spine in ax.spines.values(): spine.set_color(COLORS['grid'])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)

# Taux cumulés
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])
df_sorted = df.sort_values('timestamp')
for grp, color, label in [('control', COLORS['control'], 'Contrôle (A)'),
                            ('treatment', COLORS['treatment'], 'Traitement (B)')]:
    sub = df_sorted[df_sorted['group'] == grp]
    cumrate = sub['converted'].expanding().mean() * 100
    ax2.plot(range(len(cumrate)), cumrate.values, color=color, lw=2, label=label, alpha=0.8)
ax2.set_xlabel('Utilisateurs (cumulés)', color=COLORS['text'])
ax2.set_ylabel('Taux conversion cumulé (%)', color=COLORS['text'])
ax2.set_title('Taux cumulé', color='white')
ax2.legend(facecolor=COLORS['grid'], labelcolor='white')
ax2.tick_params(colors=COLORS['text'])
for spine in ax2.spines.values(): spine.set_color(COLORS['grid'])
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.yaxis.grid(True, color=COLORS['grid'], linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('plots/03_temporal_evolution.png', dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✅ Graphique sauvegardé : plots/03_temporal_evolution.png')

## 5. Rapport de Décision


In [ ]:
# ─── RAPPORT FINAL ──────────────────────────────────────
lift = (conv_treatment - conv_control) / conv_control * 100
is_significant = p_chi2 < ALPHA

print('=' * 60)
print('RAPPORT DE DÉCISION — A/B TEST')
print('=' * 60)
print(f'\n📌 EXPÉRIENCE')
print(f'   Contrôle : Version A (landing page originale)')
print(f'   Traitement : Version B (nouvelle landing page)')
print(f'   N total   : {len(df):,} utilisateurs')
print(f'   Durée     : {df["date"].nunique()} jours')
print()
print(f'📊 MÉTRIQUES PRINCIPALES')
print(f'   Conv. Contrôle   : {conv_control*100:.3f}%')
print(f'   Conv. Traitement : {conv_treatment*100:.3f}%')
print(f'   Lift relatif     : {lift:+.2f}%')
print()
print(f'🔬 STATISTIQUES')
print(f'   Chi² = {chi2_val:.4f} | p = {p_chi2:.6f} | α = {ALPHA}')
print(f'   Z    = {z_score:.4f} | p = {p_ztest:.6f}')
print(f'   IC {int((1-ALPHA)*100)}% diff. : [{ci_low:.5f}, {ci_high:.5f}]')
print()
print(f'✅ DÉCISION')
if is_significant and lift > 0:
    print(f'   → DÉPLOYER la Version B')
    print(f'   La version B est statistiquement supérieure (+{lift:.2f}% lift)')
elif is_significant and lift < 0:
    print(f'   → CONSERVER la Version A')
    print(f'   La version B est statistiquement inférieure ({lift:.2f}% lift)')
else:
    print(f'   → CONTINUER le test — données insuffisantes')
    print(f'   p-value ({p_chi2:.4f}) > α ({ALPHA}) — non concluant')
print('=' * 60)

In [ ]:
# ─── EXPORT ─────────────────────────────────────────────
df.to_csv('ab_data_processed.csv', index=False)
print('✅ Données exportées : ab_data_processed.csv')
print('\n🎉 Analyse terminée !')